In [ ]:
# Script to clone the repository and switch to the desired branch (ONLY FOR COLAB)

!git clone https://github.com/francescopausellii/uniform-coloring-ai.git
%cd uniform-coloring-ai
!git checkout feat/path-finding

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [3]:
# INPUT DEL PROBLEMA — due modalità alternative:
#  - INPUT_IMAGE: path di un'immagine da cui riconoscere la griglia (rete neurale)
#  - INPUT_GRID:  griglia definita a mano (usata se INPUT_IMAGE è None)
INPUT_IMAGE = "../grid_imgs/image2.png"
# INPUT_IMAGE = None

INPUT_GRID = [
    ["G", "B", "Y", "G"],
    ["Y", "G", "B", "B"],
    ["B", "Y", "G", "Y"],
    ["T", "B", "Y", "G"],
]

In [ ]:
if INPUT_IMAGE is not None:
    # Riconoscimento della griglia dall'immagine
    import keras
    from grid_recognition import (
        preparation,
        extract_grid_mask,
        detect_segments,
        classify_segments,
        segment_representative_position,
        cluster_lines_by_position,
        filter_clusters_by_span,
        build_grid_lines,
        compute_intersections,
        extract_all_cells,
        predict_grid,
        draw_all_segments,
        draw_grid_lines,
        draw_intersections,
        visualize_cells,
        print_matrix,
    )

    # Modello di riconoscimento lettere addestrato in model.ipynb
    model = keras.models.load_model("../models/dense.keras")

    # Carica l'immagine, la converte in scala di grigi e ne estrae i contorni (Canny)
    img, gray, edges = preparation(INPUT_IMAGE, show=True)

    # Rileva segmenti di linee con Hough, classificandoli in orizzontali e verticali
    grid_mask = extract_grid_mask(gray)
    segs = detect_segments(grid_mask)

    # Classifica i segmenti in orizzontali e verticali, scartando quelli obliqui
    horiz, vert = classify_segments(segs)

    # Mostra tutti i segmenti rilevati, evidenziando quelli orizzontali e verticali
    draw_all_segments(img, segs, horiz, vert)

    # Calcola la coordinata trasversale (X o Y) di ogni singolo frammento
    horiz_cl = cluster_lines_by_position(segment_representative_position(horiz, "y"))
    vert_cl = cluster_lines_by_position(segment_representative_position(vert, "x"))
    horiz_cl, vert_cl = filter_clusters_by_span(horiz_cl, vert_cl)
    h_lines, v_lines = build_grid_lines(horiz_cl, vert_cl, img.shape)
    draw_grid_lines(img, h_lines, v_lines)

    # Calcola le intersezioni tra le linee della griglia per ottenere i vertici delle celle
    pts = compute_intersections(h_lines, v_lines)
    draw_intersections(img, pts)

    # Calcola quante celle ci sono in totale (es. se ci sono 4 linee verticali, ci sono 3 colonne)
    print(f"Griglia: {pts.shape[0] - 1} righe × {pts.shape[1] - 1} colonne")

    # Ritaglia, raddrizza in prospettiva ed elimina i bordi neri da ogni quadratino
    cells_2d = extract_all_cells(gray, pts)
    visualize_cells(cells_2d)

    # Passa ogni singola immagine ritagliata alla funzione che si interfaccia con la rete neurale
    grid_matrix = predict_grid(cells_2d, model)
    print_matrix(grid_matrix)
else:
    # Griglia fornita a mano, nessun riconoscimento
    grid_matrix = INPUT_GRID
    print("Griglia fornita manualmente:", grid_matrix)

In [6]:
from uniform_coloring.problem import UniformColoring
from uniform_coloring.search import (
    uniform_cost_search as ucs,
    astar_search as astar,
)

# Problem Initialization: griglia riconosciuta dall'immagine o fornita a mano
problem = UniformColoring(grid_matrix)

In [7]:
from uniform_coloring.formatting import (
    print_section,
    print_grid,
    print_solution_results,
)

# Initial State Visualization
print_section("INITIAL STATE")
grid, pos = problem.initial.grid, problem.initial.pos
print(f"Position: {pos}")
print("Grid:")
print_grid(grid, pos)

# 2. Test BFS (Breadth-First Search)
# Optimize the number of STEPS, ignoring color costs
# print_section("BREADTH-FIRST SEARCH (Optimize Steps)")
# sol_bfs = bfs(problem)
# print_solution_results(sol_bfs, "BFS")

# 3. Test UCS (Uniform Cost Search)
# Optimize the TOTAL COST, considering the weights of colors and movements
print_section("UNIFORM COST SEARCH (Optimize Cost)")
sol_ucs = ucs(problem)
print_solution_results(sol_ucs, "UCS")


  INITIAL STATE
Position: (0, 2)
Grid:
  B  | G  | T  <- HEAD
  Y  | Y  | G 
  B  | G  | B 

  UNIFORM COST SEARCH (Optimize Cost)
--- Result UCS ---
✓ Solution found in 13 steps
Total cost = 13

Action sequence:
  1. Move.SOUTH
  2. Color.BLUE
  3. Move.WEST
  4. Color.BLUE
  5. Move.SOUTH
  6. Color.BLUE
  7. Move.WEST
  8. Move.NORTH
  9. Color.BLUE
  10. Move.EAST
  11. Move.NORTH
  12. Color.BLUE
  13. Move.EAST


In [8]:
import time
from uniform_coloring.heuristics import Heuristics

h = Heuristics(problem)
print(f"Target color: {problem.target_color.symbol}")

# A* — color
print_section("A* — color")
t0 = time.perf_counter()
sol = astar(problem, h=h.color)
print_solution_results(sol, "A* — color")
print(f"  Time: {time.perf_counter() - t0:.4f}s")

# A* — color_nearest_distance
print_section("A* — color_nearest_distance")
t0 = time.perf_counter()
sol = astar(problem, h=h.color_nearest_distance)
print_solution_results(sol, "A* — color_nearest_distance")
print(f"  Time: {time.perf_counter() - t0:.4f}s")

# A* — color_nearest_neighbor_distance (not admissible)
print_section("A* — color_nearest_neighbor_distance")
t0 = time.perf_counter()
sol = astar(problem, h=h.color_nearest_neighbor_distance)
print_solution_results(sol, "A* — color_nearest_neighbor_distance")
print(f"  Time: {time.perf_counter() - t0:.4f}s")

# A* — minimum_spanning_tree
print_section("A* — MST (Minimum Spanning Tree)")
t0 = time.perf_counter()
sol = astar(problem, h=h.mst)
print_solution_results(sol, "A* — MST")
print(f"  Time: {time.perf_counter() - t0:.4f}s")


Target color: B

  A* — color
--- Result A* — color ---
✓ Solution found in 13 steps
Total cost = 13

Action sequence:
  1. Move.SOUTH
  2. Color.BLUE
  3. Move.WEST
  4. Color.BLUE
  5. Move.SOUTH
  6. Color.BLUE
  7. Move.WEST
  8. Move.NORTH
  9. Color.BLUE
  10. Move.NORTH
  11. Move.EAST
  12. Color.BLUE
  13. Move.EAST
  Time: 0.0206s

  A* — color_nearest_distance
--- Result A* — color_nearest_distance ---
✓ Solution found in 13 steps
Total cost = 13

Action sequence:
  1. Move.WEST
  2. Color.BLUE
  3. Move.SOUTH
  4. Move.WEST
  5. Color.BLUE
  6. Move.EAST
  7. Move.SOUTH
  8. Color.BLUE
  9. Move.NORTH
  10. Color.BLUE
  11. Move.EAST
  12. Color.BLUE
  13. Move.NORTH
  Time: 0.0128s

  A* — color_nearest_neighbor_distance
--- Result A* — color_nearest_neighbor_distance ---
✓ Solution found in 13 steps
Total cost = 13

Action sequence:
  1. Move.SOUTH
  2. Color.BLUE
  3. Move.WEST
  4. Move.SOUTH
  5. Color.BLUE
  6. Move.WEST
  7. Move.NORTH
  8. Color.BLUE
  9. Move.EAST


In [9]:
# A* — ideal / TSP exact (slow on large grids)
print_section("A* — ideal (TSP exact)")
t0 = time.perf_counter()
sol = astar(problem, h=h.ideal)
print_solution_results(sol, "A* — ideal")
print(f"  Time: {time.perf_counter() - t0:.4f}s")


  A* — ideal (TSP exact)
--- Result A* — ideal ---
✓ Solution found in 13 steps
Total cost = 13

Action sequence:
  1. Move.SOUTH
  2. Color.BLUE
  3. Move.SOUTH
  4. Move.WEST
  5. Color.BLUE
  6. Move.WEST
  7. Move.NORTH
  8. Color.BLUE
  9. Move.EAST
  10. Color.BLUE
  11. Move.NORTH
  12. Color.BLUE
  13. Move.EAST
  Time: 0.0116s


In [10]:
# A* — heuristic_mismatched_coloring
print_section("A* — heuristic_mismatched_coloring")
t0 = time.perf_counter()
sol = astar(problem, h=h.heuristic_mismatched_coloring)
print_solution_results(sol, "A* — heuristic_mismatched_coloring")
print(f"  Time: {time.perf_counter() - t0:.4f}s")



  A* — heuristic_mismatched_coloring
--- Result A* — heuristic_mismatched_coloring ---
✓ Solution found in 13 steps
Total cost = 13

Action sequence:
  1. Move.SOUTH
  2. Color.BLUE
  3. Move.WEST
  4. Color.BLUE
  5. Move.SOUTH
  6. Color.BLUE
  7. Move.NORTH
  8. Move.WEST
  9. Color.BLUE
  10. Move.EAST
  11. Move.NORTH
  12. Color.BLUE
  13. Move.EAST
  Time: 0.0299s
